### Overview

Cross-validation of rank-based machine learning models using SCAN-B HiSeq training set (80%)

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
import random

#### 1. Import and prepare data

In [2]:
train_data = pd.read_csv("C:/Users/User/Documents/master_thesis_project_analysis/datasets/SCANB_GSE202203/scanb_hiseq_train_test_sets/train_test_80_20/SCANB_HiSeq_pam50gene_tpm_counts_subtype_train_80.csv", 
                          header=0, index_col=0)

In [3]:
train_data.shape

(2204, 52)

In [4]:
X_train = train_data.iloc[:, 0:50]
Y_train = train_data.iloc[:, [50]]

# check if the indices of x and y training sets match
X_train.index.equals(Y_train.index)

True

In [5]:
# label encoding of Y_train
label_encoder = LabelEncoder()
Y_train['subtype'] = label_encoder.fit_transform(Y_train['subtype'])

# check class count before label encoding
print("Class count before label encoding")
print(train_data['subtype'].value_counts())

# check class count after label encoding
print("\nClass count after label encoding")
print(Y_train['subtype'].value_counts())

Class count before label encoding
subtype
LumA     1119
LumB      654
Basal     230
Her2      201
Name: count, dtype: int64

Class count after label encoding
subtype
2    1119
3     654
0     230
1     201
Name: count, dtype: int64


C:\Users\User\AppData\Local\Temp\ipykernel_15004\4218632629.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Y_train['subtype'] = label_encoder.fit_transform(Y_train['subtype'])


In [6]:
# replace gene values by the rank
X_train_ranked = X_train.rank(axis=1, method="min", ascending=False).astype('int')

In [7]:
# scale the ranks between 0 and 1 for each sample
max_rank = len(X_train.columns)
X_train_ranked_scaled = X_train_ranked.apply(lambda row: (row-1) / (max_rank-1), axis=1)

#### 2. Cross-validation of model performance

In [8]:
seed = 42
np.random.seed(seed)
random.seed(seed)

# define stratified k fold
stratified_k_fold = StratifiedKFold(n_splits=5, shuffle = True, random_state = seed)

####  2.1 Random Forest

In [9]:
random.seed(seed)
np.random.seed(seed)

fold = 1
f1_pam50_rf = []
acc_pam50_rf = []
prec_pam50_rf = []
recall_pam50_rf = []
mcc_pam50_rf = []

for train_index, val_index in stratified_k_fold.split(Y_train, Y_train):
    print(f"Starting fold {fold}\n")
    
    # split the training set into train and validation sets for 5-fold cross validation
    print(f"Splitting training set into train and validation sets...")
    x_train, x_val = X_train_ranked_scaled.iloc[train_index], X_train_ranked_scaled.iloc[val_index]
    y_train, y_val = Y_train.iloc[train_index], Y_train.iloc[val_index]

    print(f"Shape of X_train and X_validation sets: {x_train.shape} and {x_val.shape}")
    print(f"Shape of y_train and y_validation sets: {y_train.shape} and {y_val.shape}")

    # build random forest classifier
    rfc = RandomForestClassifier(n_estimators=200, min_samples_leaf=1, min_samples_split=8, max_depth=None, random_state=seed)

    # fit the rfc model on the training fold sets
    rfc.fit(x_train, y_train.values.ravel())

    # make predictions on the validation fold set
    y_pred = rfc.predict(x_val)

    # calculate metric scores
    mcc = metrics.matthews_corrcoef(y_val.values.ravel(), y_pred)
    f1 = metrics.f1_score(y_val.values.ravel(), y_pred, average='macro')
    recall = metrics.recall_score(y_val.values.ravel(), y_pred, average='macro')
    precision = metrics.precision_score(y_val.values.ravel(), y_pred, average='macro')
    accuracy = metrics.accuracy_score(y_val.values.ravel(), y_pred)
    
    print(metrics.classification_report(y_val.values.ravel(), y_pred, digits=4))

    mcc_pam50_rf.append(mcc)
    f1_pam50_rf.append(f1)
    recall_pam50_rf.append(recall)
    prec_pam50_rf.append(precision)
    acc_pam50_rf.append(accuracy)

    fold += 1


Starting fold 1

Splitting training set into train and validation sets...
Shape of X_train and X_validation sets: (1763, 50) and (441, 50)
Shape of y_train and y_validation sets: (1763, 1) and (441, 1)
              precision    recall  f1-score   support

           0     1.0000    0.9565    0.9778        46
           1     1.0000    0.8500    0.9189        40
           2     0.9685    0.9598    0.9641       224
           3     0.8865    0.9542    0.9191       131

    accuracy                         0.9478       441
   macro avg     0.9637    0.9301    0.9450       441
weighted avg     0.9503    0.9478    0.9481       441

Starting fold 2

Splitting training set into train and validation sets...
Shape of X_train and X_validation sets: (1763, 50) and (441, 50)
Shape of y_train and y_validation sets: (1763, 1) and (441, 1)
              precision    recall  f1-score   support

           0     0.9783    0.9783    0.9783        46
           1     0.9375    0.7500    0.8333        4

In [10]:
print(f'Mean Accuracy: {round(np.mean(acc_pam50_rf), 4)} \u00B1 {round(np.std(acc_pam50_rf), 4)}')
print(f'Mean Precision: {round(np.mean(prec_pam50_rf), 4)} \u00B1  {round(np.std(prec_pam50_rf), 4)}')
print(f'Mean Recall: {round(np.mean(recall_pam50_rf), 4)} \u00B1  {round(np.std(recall_pam50_rf), 4)}')
print(f'Mean F1: {round(np.mean(f1_pam50_rf), 4)} \u00B1  {round(np.std(f1_pam50_rf), 4)}')
print(f'Mean MCC: {round(np.mean(mcc_pam50_rf), 4)} \u00B1  {round(np.std(mcc_pam50_rf), 4)}')

Mean Accuracy: 0.9365 ± 0.0099
Mean Precision: 0.95 ±  0.0139
Mean Recall: 0.9188 ±  0.015
Mean F1: 0.9327 ±  0.0147
Mean MCC: 0.8996 ±  0.0158


#### 2.2 SVM

In [11]:
random.seed(seed)
np.random.seed(seed)

fold = 1
f1_pam50_svm = []
acc_pam50_svm = []
prec_pam50_svm = []
recall_pam50_svm = []
mcc_pam50_svm = []

for train_index, val_index in stratified_k_fold.split(Y_train, Y_train):
    print(f"Starting fold {fold}\n")
    
    # split the training set into train and validation sets for 5-fold cross validation
    print(f"Splitting training set into train and validation sets...")
    x_train, x_val = X_train_ranked_scaled.iloc[train_index], X_train_ranked_scaled.iloc[val_index]
    y_train, y_val = Y_train.iloc[train_index], Y_train.iloc[val_index]

    print(f"Shape of X_train and X_validation sets: {x_train.shape} and {x_val.shape}")
    print(f"Shape of y_train and y_validation sets: {y_train.shape} and {y_val.shape}")

    # build logistic regression classifier
    svm = SVC(C=1, gamma=1, kernel='rbf', random_state=seed)

    # fit the logistic regression model on the training fold sets
    svm.fit(x_train, y_train.values.ravel())

    # make predictions on the validation fold set
    y_pred = svm.predict(x_val)

    # calculate metric scores
    mcc = metrics.matthews_corrcoef(y_val.values.ravel(), y_pred)
    f1 = metrics.f1_score(y_val.values.ravel(), y_pred, average='macro')
    recall = metrics.recall_score(y_val.values.ravel(), y_pred, average='macro')
    precision = metrics.precision_score(y_val.values.ravel(), y_pred, average='macro')
    accuracy = metrics.accuracy_score(y_val.values.ravel(), y_pred)
    
    print(metrics.classification_report(y_val.values.ravel(), y_pred, digits=4))

    mcc_pam50_svm.append(mcc)
    f1_pam50_svm.append(f1)
    recall_pam50_svm.append(recall)
    prec_pam50_svm.append(precision)
    acc_pam50_svm.append(accuracy)

    fold += 1


Starting fold 1

Splitting training set into train and validation sets...
Shape of X_train and X_validation sets: (1763, 50) and (441, 50)
Shape of y_train and y_validation sets: (1763, 1) and (441, 1)
              precision    recall  f1-score   support

           0     1.0000    0.9348    0.9663        46
           1     0.9722    0.8750    0.9211        40
           2     0.9511    0.9554    0.9532       224
           3     0.8832    0.9237    0.9030       131

    accuracy                         0.9365       441
   macro avg     0.9516    0.9222    0.9359       441
weighted avg     0.9380    0.9365    0.9367       441

Starting fold 2

Splitting training set into train and validation sets...
Shape of X_train and X_validation sets: (1763, 50) and (441, 50)
Shape of y_train and y_validation sets: (1763, 1) and (441, 1)
              precision    recall  f1-score   support

           0     0.9778    0.9565    0.9670        46
           1     0.9189    0.8500    0.8831        4

In [12]:
print(f'Mean Accuracy: {round(np.mean(acc_pam50_svm), 4)} \u00B1 {round(np.std(acc_pam50_svm), 4)}')
print(f'Mean Precision: {round(np.mean(prec_pam50_svm), 4)} \u00B1  {round(np.std(prec_pam50_svm), 4)}')
print(f'Mean Recall: {round(np.mean(recall_pam50_svm), 4)} \u00B1  {round(np.std(recall_pam50_svm), 4)}')
print(f'Mean F1: {round(np.mean(f1_pam50_svm), 4)} \u00B1  {round(np.std(f1_pam50_svm), 4)}')
print(f'Mean MCC: {round(np.mean(mcc_pam50_svm), 4)} \u00B1  {round(np.std(mcc_pam50_svm), 4)}')

Mean Accuracy: 0.9401 ± 0.0071
Mean Precision: 0.947 ±  0.0121
Mean Recall: 0.93 ±  0.0092
Mean F1: 0.9379 ±  0.01
Mean MCC: 0.9056 ±  0.0112


#### 2.3 Logistic Regression

In [13]:
random.seed(seed)
np.random.seed(seed)

fold = 1
f1_pam50_lr = []
acc_pam50_lr = []
prec_pam50_lr = []
recall_pam50_lr = []
mcc_pam50_lr = []

for train_index, val_index in stratified_k_fold.split(Y_train, Y_train):
    print(f"Starting fold {fold}\n")
    
    # split the training set into train and validation sets for 5-fold cross validation
    print(f"Splitting training set into train and validation sets...")
    x_train, x_val = X_train_ranked_scaled.iloc[train_index], X_train_ranked_scaled.iloc[val_index]
    y_train, y_val = Y_train.iloc[train_index], Y_train.iloc[val_index]

    print(f"Shape of X_train and X_validation sets: {x_train.shape} and {x_val.shape}")
    print(f"Shape of y_train and y_validation sets: {y_train.shape} and {y_val.shape}")

    # build logistic regression classifier
    lr = LogisticRegression(C=5, penalty='l2', solver='lbfgs', random_state=seed, max_iter=2000)

    # fit the logistic regression model on the training fold sets
    lr.fit(x_train, y_train.values.ravel())
    
    # make predictions on the validation fold set
    y_pred = lr.predict(x_val)

    # calculate metric scores
    mcc = metrics.matthews_corrcoef(y_val.values.ravel(), y_pred)
    f1 = metrics.f1_score(y_val.values.ravel(), y_pred, average='macro')
    recall = metrics.recall_score(y_val.values.ravel(), y_pred, average='macro')
    precision = metrics.precision_score(y_val.values.ravel(), y_pred, average='macro')
    accuracy = metrics.accuracy_score(y_val.values.ravel(), y_pred)
    
    print(metrics.classification_report(y_val.values.ravel(), y_pred, digits=4))

    mcc_pam50_lr.append(mcc)
    f1_pam50_lr.append(f1)
    recall_pam50_lr.append(recall)
    prec_pam50_lr.append(precision)
    acc_pam50_lr.append(accuracy)

    fold += 1


Starting fold 1

Splitting training set into train and validation sets...
Shape of X_train and X_validation sets: (1763, 50) and (441, 50)
Shape of y_train and y_validation sets: (1763, 1) and (441, 1)
              precision    recall  f1-score   support

           0     1.0000    0.9565    0.9778        46
           1     0.9730    0.9000    0.9351        40
           2     0.9469    0.9554    0.9511       224
           3     0.8955    0.9160    0.9057       131

    accuracy                         0.9388       441
   macro avg     0.9538    0.9320    0.9424       441
weighted avg     0.9395    0.9388    0.9389       441

Starting fold 2

Splitting training set into train and validation sets...
Shape of X_train and X_validation sets: (1763, 50) and (441, 50)
Shape of y_train and y_validation sets: (1763, 1) and (441, 1)
              precision    recall  f1-score   support

           0     0.9783    0.9783    0.9783        46
           1     0.8205    0.8000    0.8101        4

In [14]:
print(f'Mean Accuracy: {round(np.mean(acc_pam50_lr), 4)} \u00B1 {round(np.std(acc_pam50_lr), 4)}')
print(f'Mean Precision: {round(np.mean(prec_pam50_lr), 4)} \u00B1  {round(np.std(prec_pam50_lr), 4)}')
print(f'Mean Recall: {round(np.mean(recall_pam50_lr), 4)} \u00B1  {round(np.std(recall_pam50_lr), 4)}')
print(f'Mean F1: {round(np.mean(f1_pam50_lr), 4)} \u00B1  {round(np.std(f1_pam50_lr), 4)}')
print(f'Mean MCC: {round(np.mean(mcc_pam50_lr), 4)} \u00B1  {round(np.std(mcc_pam50_lr), 4)}')

Mean Accuracy: 0.9424 ± 0.0106
Mean Precision: 0.944 ±  0.0218
Mean Recall: 0.9326 ±  0.0122
Mean F1: 0.938 ±  0.0167
Mean MCC: 0.9093 ±  0.0164
